#Lab 7: 3D Stokes problem

The following example concerns the FE approximation of a 3D Stokes problem with mixed type boundary condition. The implementation is inspired by the DOLFIN demo [available here](https://bitbucket.org/fenics-project/dolfin/src/master/python/demo/documented/stokes-iterative/demo_stokes-iterative.py.rst). The discretization is based on the **Taylor-Hood element pair** and the algebraic solver resort to a **preconditioned Krylov solver**.

## FEniCS implementation

In [0]:
%%capture
!apt-get install -y -qq software-properties-common python-software-properties module-init-tools
!add-apt-repository -y ppa:fenics-packages/fenics
!apt-get update -qq
!apt install -y --no-install-recommends fenics
!rm -rf *
from fenics import *

First, a uniform mesh of the unit cube is generated. Then, the "mixed" element corresponding to the lowest-order stable Taylor-Hood coupling is defined.

In [3]:
# Load mesh
mesh = UnitCubeMesh(16, 16, 16)
# also hexaedrale elements are supported: 
# mesh = UnitCubeMesh.create(16, 16, 16, CellType.Type.hexahedron)
print(mesh.ufl_cell())

# Build function space
P2 = VectorElement("Lagrange", mesh.ufl_cell(), 2)
P1 = FiniteElement("Lagrange", mesh.ufl_cell(), 1)
TH = P2 * P1
W = FunctionSpace(mesh, TH)

tetrahedron


Next, the boundary conditions are enforced and the bilinear and linear forms corresponding to the weak formulation are defined.

In [0]:
# Boundaries
def right(x, on_boundary):
  return x[0] > (1.0 - DOLFIN_EPS)
def top_bottom(x, on_boundary):
  return x[1] > 1.0 - DOLFIN_EPS or x[1] < DOLFIN_EPS

# No-slip boundary condition for velocity
noslip = Constant((0.0, 0.0, 0.0))
bc0 = DirichletBC(W.sub(0), noslip, top_bottom)

# Inflow boundary condition for velocity
inflow = Expression(("-sin(x[1]*pi)", "0.0", "0.0"), degree=2)
bc1 = DirichletBC(W.sub(0), inflow, right)

# Collect boundary conditions
bcs = [bc0, bc1]

# Define variational problem
(u, p) = TrialFunctions(W)
(v, q) = TestFunctions(W)
f = Constant((0.0, 0.0, 0.0))
a = inner(grad(u), grad(v))*dx + div(v)*p*dx + q*div(u)*dx
L = inner(f, v)*dx

Now we define the form corresponding to the expression for the preconditioner and we assemble the matrix corresponding to the bilinear form and the vector corresponding to the linear form of the Stokes equations, applying also the specified boundary conditions to the linear system.

We perform the assembly for the preconditioner matrix P in the same way using the linear form L as a dummy form.

In [0]:
# Form for use in constructing preconditioner matrix
b = inner(grad(u), grad(v))*dx + p*q*dx

# Assemble system
A, bb = assemble_system(a, L, bcs)

# Assemble preconditioner system
P, btmp = assemble_system(b, L, bcs)

Finally, we specify the iterative solver we want to use, solve the linear system, and postprocess the results.

In [0]:
# Solution of the linear system
U = Function(W)

# Create Krylov solver and AMG preconditioner
solver = KrylovSolver("default", "amg")

# Associate operator (A) and preconditioner matrix (P)
solver.set_operators(A, P)

# Solve
solver.solve(U.vector(), bb)

# Get sub-functions
u, p = U.split()

# Save solution in VTK format
ufile_pvd = File("velocity.pvd")
ufile_pvd << u
pfile_pvd = File("pressure.pvd")
pfile_pvd << p